# Scraping AWOIAF Character Biographies
Produces `characters_bio.csv` with columns: `name`, `ID`, `bio`

- **`bio`**: The full narrative text from the character's wiki page (their story history), with all recognised character names replaced by their canonical `ID` slug.
- This makes the bio machine-readable for downstream tasks like relationship sentiment extraction.

Requires `characters.csv` from the original `scrape_characters.ipynb` (or will re-scrape the list if not found).

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import re
import os
from urllib.parse import quote, unquote
from concurrent.futures import ThreadPoolExecutor

## Setup

In [2]:
BASE   = 'https://awoiaf.westeros.org'
PREFIX = '/index.php/'

headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

session = requests.Session()
session.headers.update(headers)

## 1. Load or scrape the character list
Reuses `characters.csv` from the original notebook if it exists.

In [3]:
CHARACTERS_CSV = 'characters.csv'

def scrape_character_list():
    list_page = session.get(BASE + PREFIX + 'List_of_characters', timeout=20)
    print(f'List page status: {list_page.status_code}')
    soup = BeautifulSoup(list_page.content, 'html.parser')
    content = soup.find('div', class_='mw-parser-output')
    characters = []
    seen_ids = set()
    for li in content.find_all('li'):
        a = li.find('a')
        if not a:
            continue
        href = a.get('href', '')
        if not href.startswith(PREFIX):
            continue
        if 'redlink=1' in href or 'action=' in href:
            continue
        slug = href[len(PREFIX):].split('#', 1)[0]
        if not slug or slug.startswith('Special:'):
            continue
        slug = unquote(slug)
        if slug in seen_ids:
            continue
        seen_ids.add(slug)
        name = a.get_text(strip=True)
        if not name:
            continue
        characters.append({'name': name, 'ID': slug})
    return characters

if os.path.exists(CHARACTERS_CSV):
    char_df = pd.read_csv(CHARACTERS_CSV)
    print(f'Loaded {len(char_df)} characters from {CHARACTERS_CSV}')
else:
    print(f'{CHARACTERS_CSV} not found — scraping character list...')
    chars = scrape_character_list()
    char_df = pd.DataFrame(chars, columns=['name', 'ID'])
    char_df.to_csv(CHARACTERS_CSV, index=False)
    print(f'Saved {len(char_df)} characters to {CHARACTERS_CSV}')

characters = char_df.to_dict('records')
print(characters[:5])

Loaded 3690 characters from characters.csv
[{'name': 'A certain man', 'ID': 'A_certain_man'}, {'name': 'Abelar Hightower', 'ID': 'Abelar_Hightower'}, {'name': 'Abelon', 'ID': 'Abelon'}, {'name': 'Addam of Duskendale', 'ID': 'Addam_of_Duskendale'}, {'name': 'Addam Frey', 'ID': 'Addam_Frey'}]


## 2. Build slug lookup set

We only need the set of valid character IDs to validate link targets.
Name → ID translation happens **at the HTML link level** during bio extraction:
if a name is hyperlinked, we use the link's target slug; if it's plain text, we leave it alone.
This avoids the Aegon Frey problem — two characters share a display name but each page
links to the correct disambiguated slug.

In [4]:
character_ids = set(row['ID'] for row in characters)
print(f'Character ID set built: {len(character_ids)} entries.')

Character ID set built: 3690 entries.


## 3. Helpers: URL utilities, link-based ID substitution & bio extraction

In [5]:
def slug_to_url(slug):
    return BASE + PREFIX + quote(slug, safe="_/(),'")


def href_to_slug(href):
    """Return the canonical (URL-decoded) slug for an internal wiki link, or None."""
    if not href.startswith(PREFIX):
        return None
    if 'redlink=1' in href or 'action=' in href:
        return None
    slug = href[len(PREFIX):].split('#', 1)[0]
    if not slug or slug.startswith('Special:'):
        return None
    return unquote(slug)


# Sections that are NOT biography / history — skip these entirely
SKIP_SECTIONS = {
    'references', 'notes', 'external links', 'see also',
    'navigation', 'appendix', 'citations', 'gallery',
    'quotes', 'tv adaptation', 'behind the scenes',
}

# Sections that ARE the biography/history — prefer these if present
BIO_SECTIONS = {
    'history', 'biography', 'background', 'recent events',
    'life', 'character and appearance',
}


def para_to_text(p, character_ids):
    """
    Convert a <p> element to a string, replacing linked character names with their
    target slug ID — but ONLY when the name is inside an <a href> that points to a
    known character page. Plain (unlinked) text is kept verbatim.

    This is the key fix: two characters named 'Aegon Frey' each link to different
    slugs (Aegon_Frey_(son_of_Aenys) vs Aegon_Frey_(son_of_Robb)), so we trust the
    link target rather than trying to match the display name.
    """
    parts = []
    for node in p.descendants:
        # Only process leaf text nodes (NavigableString), not tag nodes
        if node.name is not None:
            continue
        text = str(node)
        if not text:
            continue
        parent = node.parent
        # Check if this text node is the direct content of an <a> tag
        if parent and parent.name == 'a':
            href = parent.get('href', '')
            slug = href_to_slug(href)
            if slug and slug in character_ids:
                # Replace the linked display text with the target character ID
                parts.append(slug)
                continue
        # Not a character link — keep the raw text
        parts.append(text)
    return ''.join(parts)


def extract_bio(soup, character_ids):
    """
    Extract the narrative biography text from a character page.

    Strategy:
    1. Find the article body (mw-parser-output).
    2. Walk headings to identify sections.
    3. Collect paragraphs from biography/history sections, converting
       linked character names to their ID slug in-place.
       If no bio sections found, fall back to ALL non-skip-section paragraphs.
    4. Strip citation markers like [1], [2].
    """
    content = soup.find('div', class_='mw-parser-output')
    if content is None:
        return ''

    elements = list(content.children)
    bio_paragraphs = []
    all_paragraphs = []
    current_section = None
    in_bio_section = False

    for el in elements:
        if not hasattr(el, 'name') or el.name is None:
            continue

        if el.name in ('h2', 'h3', 'h4'):
            heading_text = el.get_text(' ', strip=True).lower()
            heading_text = re.sub(r'\[edit\]', '', heading_text).strip()
            current_section = heading_text
            in_bio_section = any(kw in current_section for kw in BIO_SECTIONS)
            continue

        if el.name != 'p':
            continue

        # Convert paragraph HTML → text with linked character names → IDs
        text = para_to_text(el, character_ids)
        # Strip citation markers
        text = re.sub(r'\[\d+\]', '', text)
        text = re.sub(r'\[nb \d+\]', '', text)
        text = text.strip()

        if not text or len(text) < 30:
            continue

        if current_section and any(kw in current_section for kw in SKIP_SECTIONS):
            continue

        all_paragraphs.append(text)
        if current_section is None or in_bio_section:
            bio_paragraphs.append(text)

    result = bio_paragraphs if bio_paragraphs else all_paragraphs
    return ' '.join(result)

## 4. Per-character scrape

In [6]:
def scrape_bio(row):
    """
    Fetch one character page and return {'name', 'ID', 'bio'}.

    Character name → ID substitution happens at the HTML link level inside
    extract_bio: only names that are hyperlinked on the page are replaced with
    their target slug. Unlinked plain text is kept verbatim, so a character
    referring to themselves by name is never wrongly replaced with a different
    disambiguated ID (e.g. Aegon Frey won't become Aegon_Frey_(son_of_Aenys)
    when the page is about Aegon_Frey_(son_of_Robb)).
    """
    name, slug = row['name'], row['ID']
    try:
        resp = session.get(slug_to_url(slug), timeout=20)
    except requests.RequestException as e:
        print(f'  ! request failed for {slug}: {e}')
        return {'name': name, 'ID': slug, 'bio': ''}

    if resp.status_code != 200:
        print(f'  ! HTTP {resp.status_code} for {slug}')
        return {'name': name, 'ID': slug, 'bio': ''}

    soup = BeautifulSoup(resp.content, 'html.parser')
    bio = extract_bio(soup, character_ids)

    return {'name': name, 'ID': slug, 'bio': bio}

## 5. Run the scrape
Parallelised across 8 threads, checkpoints every 100 rows.

In [7]:
OUT     = 'characters_bio.csv'
COLUMNS = ['name', 'ID', 'bio']
WORKERS = 8

rows = []
with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    for i, row in enumerate(executor.map(scrape_bio, characters), 1):
        rows.append(row)
        if i % 50 == 0:
            print(f'  {i}/{len(characters)}')
        if i % 100 == 0:
            pd.DataFrame(rows, columns=COLUMNS).to_csv(OUT, index=False)

pd.DataFrame(rows, columns=COLUMNS).to_csv(OUT, index=False)
print(f'\nSaved {len(rows)} rows to {OUT}')

  50/3690
  100/3690
  150/3690
  200/3690
  250/3690
  300/3690
  350/3690
  400/3690
  450/3690
  500/3690
  550/3690
  600/3690
  650/3690
  700/3690
  750/3690
  800/3690
  850/3690
  900/3690
  950/3690
  1000/3690
  1050/3690
  1100/3690
  1150/3690
  1200/3690
  1250/3690
  1300/3690
  1350/3690
  1400/3690
  1450/3690
  1500/3690
  1550/3690
  1600/3690
  1650/3690
  1700/3690
  1750/3690
  1800/3690
  1850/3690
  1900/3690
  1950/3690
  2000/3690
  2050/3690
  2100/3690
  2150/3690
  2200/3690
  2250/3690
  2300/3690
  2350/3690
  2400/3690
  2450/3690
  2500/3690
  2550/3690
  2600/3690
  2650/3690
  2700/3690
  2750/3690
  2800/3690
  2850/3690
  2900/3690
  2950/3690
  3000/3690
  3050/3690
  3100/3690
  3150/3690
  3200/3690
  3250/3690
  3300/3690
  3350/3690
  3400/3690
  3450/3690
  3500/3690
  3550/3690
  3600/3690
  3650/3690

Saved 3690 rows to characters_bio.csv


## 6. Sanity check

In [8]:
df = pd.read_csv(OUT)
print(f'Shape: {df.shape}')
print(f'Empty bios: {(df["bio"] == "").sum()}')
print(f'Median bio length (chars): {df["bio"].str.len().median():.0f}')
print()

# Inspect a well-known character
sample = df[df['name'] == 'Eddard Stark']
if not sample.empty:
    print('--- Eddard Stark bio (first 800 chars) ---')
    print(sample.iloc[0]['bio'][:800])

Shape: (3690, 3)
Empty bios: 0
Median bio length (chars): 242

--- Eddard Stark bio (first 800 chars) ---
Eddard Stark, also called "Ned", is the head of House Stark, Lord of Winterfell, and Warden of the North. He is a close friend to King Robert_I_Baratheon, with whom he was raised. Eddard is one of the major POV characters in A Song of Ice and Fire. In the television adaptation Game of Thrones, Eddard is portrayed by Sean Bean, Sebastian Croft (as a child), and Robert Aramayo (as a young man).


---
## Notes for downstream sentiment / relationship extraction

The `bio` column now contains prose where every recognised character name has been
replaced with their unique `ID` slug (e.g. `Eddard_Stark`, `Jon_Snow`).

**Suggested next steps:**

1. **Sentence-level co-occurrence**: Split each bio into sentences, find pairs of IDs that appear together, and label the sentence as positive/negative/neutral.
2. **LLM extraction**: For each character, pass their bio to a model with a prompt like:
   > *"For each character ID mentioned in this biography, classify the subject's relationship to them as: positive / negative / neutral. Output JSON."*
3. **Build an edge list**: Combine all (source_ID, target_ID, sentiment) triples into a signed graph for network analysis.

The ID substitution ensures that even highly ambiguous names ("Jon", "Aegon") map to
unambiguous nodes in the graph.